# 統計模型與損失函數

對應 DLI：01、02。

本課使用小型合成資料或 scikit-learn 內建資料，在 CPU 執行，不需下載原課程資料。依序執行所有儲存格。

[在 Google Colab 執行本課](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/05_statistical_models_loss.ipynb) · [閱讀完整背景講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/05-statistical-models-loss.md)

建議順序：先讀概念與手算 → 回答檢核 → 執行程式 → 修改練習。

[背景導讀：從統計到 AI 的模型設計](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/statistics-to-ai.md)

## 從圖開始

![單筆誤差 2 與 4 ms 時，MAE 貢獻加倍，MSE 貢獻變成四倍](https://raw.githubusercontent.com/Samuel-ncu/AI-Stastic/main/docs/figures/05-loss.svg)

圖中畫的是單筆誤差的貢獻。先算 2 ms 與 4 ms 的絕對及平方損失，接著用同一套步驟讀取迴歸殘差。 [閱讀本課講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/05-statistical-models-loss.md)

## 學習目標與先備知識

先讀條件機率與標準化。需要理解函數是「輸入數值得到輸出」；本課補充指數與對數，不要求先學最佳化推導。



## 補充｜ 從一條直線開始預測延遲

假設你要預測一套影像辨識服務的處理時間。輸入 $x$ 是排在前面的請求數，輸出 $y$ 是量到的延遲（毫秒）。本章的資料、數字與練習都是教學示例。

把一條直線當作第一個模型：

$$\hat y=b+wx$$

$\hat y$ 是模型的預測，$b$ 是截距，代表 $x=0$ 時模型的預測。$w$ 是斜率，描述模型預測裡，每多一個請求，延遲改變幾毫秒。這個斜率來自資料的擬合，若要解讀為因果效果，還需要合適的實驗或辨識條件。

### 殘差是圖上哪一段？

在相同的 $x_i$，從直線上的預測值走到實際資料點，垂直距離帶正負號就是殘差 $r_i=y_i-\hat y_i$。正值代表低估，負值代表高估。這是用已觀測的答案與已擬合模型算出的差，理論模型中的隨機誤差則是相對於未知的真實條件平均。

| 前方請求數 $x$ | 實測 $y$（ms） | 起始直線 $\hat y=2x$ | 殘差 $r$（ms） | 殘差平方（ms²） |
|---:|---:|---:|---:|---:|
| 1 | 2 | 2 | 0 | 0 |
| 2 | 3 | 4 | −1 | 1 |
| 3 | 7 | 6 | 1 | 1 |

這三個殘差的總和是 0，但只有第一筆預測準確。若直接加總，低估和高估會抵銷。平方後再平均，得到 $MSE=(0+1+1)/3\approx0.667$ ms²。MSE 把大偏差加重。例如 4 ms 的偏差是 2 ms 的兩倍，平方懲罰卻是四倍。

### 迴歸怎麼選出直線？

普通最小平方法（OLS）選擇 $b,w$，使 $\sum_i(y_i-b-wx_i)^2$ 最小。同一批資料上，除以固定的筆數只會改變數值尺度，因此使 SSE 最小的直線也會使 MSE 最小。這個例子的最小平方直線是 $\hat y=-1+2.5x$，預測為 $[1.5,4,6.5]$，殘差為 $[0.5,-1,0.5]$，MSE 為 0.5 ms²。截距為負也提醒我們，模型在觀察範圍外的預測需要另外驗證。

**進一步手算。** $\bar x=2$，$\bar y=4$。斜率是 $\sum_i(x_i-\bar x)(y_i-\bar y)/\sum_i(x_i-\bar x)^2=5/2$，截距是 $\bar y-w\bar x=4-5=-1$。對新資料能否預測得好，要看留出的資料與殘差圖。若殘差在某個區間總是偏正，先檢查模型是否漏掉轉折或重要變數，再考慮改模型。[來源：NIST 線性最小平方迴歸](https://www.itl.nist.gov/div898/handbook/pmd/section1/pmd141.htm)

**自我檢核。** 若預測 9 ms，實測 12 ms，殘差為 $+3$ ms，代表低估。重新擬合後，各點的殘差都可能改變。接著可看 [第 06 課的 ReLU 轉折](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/06-overfitting-generalization.md) 與 [第 10 課的兩種殘差比較](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/10-text-statistics.md)。

## 1. 從猜一個數字到猜一個類別

**來源摘要 [S5](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)：**線性模型結合特徵與係數；邏輯斯迴歸用於分類，可以產生類別機率。名稱中雖有「迴歸」，用途仍可為分類。

### 二元邏輯斯迴歸（Binary Logistic Regression）

前面的線性迴歸用排在前面的請求數 $x$ 預測延遲幾毫秒。現在改問「延遲是否超過 12 ms？」。用 $y=0$ 表示未超時，$y=1$ 表示超時。二元邏輯斯迴歸是描述**給定 $x$ 時，事件發生機率** $p=P(y=1\mid x)$ 的統計模型。

二元邏輯斯迴歸先用 $z=b+wx$ 算一個分數。加權分數可以是任意實數，因此再用 **Sigmoid** $\sigma(z)=1/(1+e^{-z})$ 把分數轉到 0 和 1 之間。分數越高，超時機率越高。未超時的機率則是 $1-p$。

為了手算，先指定截距 $b=-4\ln 2$ 和斜率 $w=\ln 2$，得到 $z=(x-4)\ln 2$。係數給定後，把 $x$ 代入公式就能算出模型的機率。

![二元邏輯斯迴歸兩圖共用請求數橫軸。左圖是線性分數，右圖是 S 形機率曲線。4 個請求對應分數 0、超時機率 50%，6 個請求對應分數約 1.39、超時機率 80%。](https://raw.githubusercontent.com/Samuel-ncu/AI-Stastic/main/docs/figures/05-binary-logistic-regression.svg)

兩圖的橫軸都是前方請求數。沿著橘色標記讀 $x=6$，左圖的分數是 $z=\ln 4\approx1.39$，經 Sigmoid 換算後，右圖的超時機率是 80%。

| 前方請求數 $x$ | 線性分數 $z$ | 超時機率 $p$ |
|---:|---:|---:|
| 3 | $-\ln 2$ | $1/(1+2)\approx33.3\%$ |
| 4 | $0$ | $1/(1+1)=50\%$ |
| 6 | $2\ln 2=\ln 4$ | $1/(1+1/4)=80\%$ |

每多一個請求，分數就增加固定的 $\ln 2$，機率增加的幅度會隨原本的位置改變。下方的 Logit 小節會說明為什麼把機率換到「對數勝算」的尺度後，就能用直線來描述。

### 多項邏輯斯迴歸（Multinomial Logistic Regression）

如果想同時區分「正常、超時、失敗」，可以讓每筆請求只屬於一類。例如先把有執行錯誤的請求標成「失敗」，其餘請求再依有沒有超過 12 ms 分成「超時」或「正常」。這就是三個互斥類別的問題。

模型讓每一類有自己的斜率和截距。給定同一筆輸入 $x$，第 $k$ 類的分數是 $z_k=b_k+w_kx$。例如下列係數會在 $x=6$ 時給出三個分數：

| 類別 | 分數公式 | $x=6$ 時的分數 |
|---|---|---:|
| 正常 | $5-0.5x$ | 2 |
| 超時 | $-2+0.5x$ | 1 |
| 失敗 | $0$（選為參考類別） | 0 |

「超時」的正斜率表示每多一個請求，它相對於參考類別的分數差會增加 0.5。**Softmax** 把各類分數一起轉成加總為 1 的機率。此例約為正常 66.5%、超時 24.5%、失敗 9.0%。失敗類的分數定為 0，它依然有機率。完整手算放在下方。

### 模型的係數怎麼從樣本估計？

上面先指定係數，示範公式怎麼算。若想從實際服務估計係數，就蒐集一批觀測樣本。每筆記下當時排在前面的請求數 $x$，以及實際結果 $y$。統計上，把依據這些樣本估計係數的步驟稱為**擬合（fit）**。機器學習把用來擬合模型的那批樣本稱為**訓練資料**。

前面用最小平方來擬合延遲的直線。邏輯斯迴歸則常用**最大概似法**，尋找能讓整批已觀測結果在模型下最可能出現的係數。在第 3 節的設定下，把概似取負對數後，二元模型可用二元交叉熵（BCE）擬合，多項模型可用多類別交叉熵擬合。

第 3 節會用 10 筆請求紀錄手算這個過程，從標準化輸入到預測新請求的機率。

二元模型先算一個分數，多項模型先算一組分數。接下來用 **logit** 和 **logits** 說明這些分數的尺度。

### Logit 是什麼？

**Logit 是分類模型在轉成機率之前給出的原始分數。** 延續超時案例，假設最後一層收到前面算出的特徵 $x$，先算 $z=b+wx$。$x$ 是模型輸入這一層的數值，$w$、$b$ 是學到的權重與偏差，$z$ 就是這筆資料的 logit。神經網路的最後一層可以同時使用多個特徵，仍是先把特徵加權、加偏差，再得到分類分數。

#### 一個 logit 如何變成二元機率？

令 $p$ 為模型給「超時」的機率，未超時的機率就是 $1-p$。比值 $p/(1-p)$ 叫做**勝算（odds）**。例如 $p=0.8$ 時，超時機率是未超時的 4 倍。取自然對數後，這個勝算轉成實數，叫做**對數勝算（log-odds）**：

$z=\ln\frac{p}{1-p},\qquad p=\sigma(z)=\frac{1}{1+e^{-z}}.$

左式是從機率回推 logit，右式是從模型分數算出機率。**二元邏輯斯迴歸擬合的是對數勝算的直線** $\ln[p/(1-p)]=b+wx$。前方每多一個請求，對數勝算增加 $w$，勝算就乘上 $e^w$。在上例中 $w=\ln 2$，所以勝算每次乘以 2。從 $x=4$ 到 $x=6$，勝算由 1 變成 4，超時機率就由 $1/(1+1)=50\%$ 變成 $4/(4+1)=80\%$。

正分數表示模型偏向超時，負分數表示偏向未超時。若要採取行動，才另外依任務選閾值。下表示範用 0.5 作為判斷閾值：

| 模型原始分數 $z$ | Sigmoid 算出的超時機率 $p$ | 採用 $p\geq0.5$ 的判斷 |
|---:|---:|---|
| $-\ln 4\approx-1.386$ | 20% | 判未超時 |
| $0$ | 50% | 判超時 |
| $\ln 4\approx1.386$ | 80% | 判超時 |
| $-8$ | 約 0.0335% | 判未超時 |

同一筆資料有三個不同的量：**模型算出的分數 $z$、從分數換算的機率 $p$、以及觀測到的實際標籤 $y$**。如果最後一筆實際超時，$y=1$，模型仍可能給 $z=-8$。預測機率極低，這筆就會得到很大的二元交叉熵。後面會說明損失怎麼算，[第 06 課的梯度例子](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/06-overfitting-generalization.md#二元分類最後一層為什麼仍用-sigmoid)會沿用同一筆資料。

#### 為什麼多類別會說 logits？

若「正常、超時、失敗」三類互斥，模型會為每一類給出一個原始分數。這一組分數就是 **logits**。

#### Softmax 的公式

假設有 $K$ 個互斥類別，$z_k$ 是模型給第 $k$ 類的 logit。Softmax 把這些分數換成機率：

$p_k=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}},\qquad k=1,\ldots,K.$

分子只用第 $k$ 類的分數，分母則將**所有類別**的分數取指數後加總。每個指數都大於 0，且各類共用同一個分母，所以 $p_1+\cdots+p_K=1$。同一組分數中，某一類的 logit 上升，分給它的機率就會增加，其他類分到的比例則會下降。

沿用「正常、超時、失敗」三選一，對 logits $[2,1,0]$ 取指數，得到約 $[7.389,2.718,1.000]$，總和約為 11.107。各類再除以 11.107，得到約 $[0.665,0.245,0.090]$。例如超時排在第二類，其機率是 $e^1/(e^2+e^1+e^0)\approx0.245$。

把所有 logits 一起加上同一個常數，分子與分母會同乘一個因子，機率保持相同。因此可以把每個分數都減去參考類別的分數。這樣參考類別的分數就成為 0，各類機率保持相同。令 $r$ 為參考類別，兩個機率相除時會消去共同分母，因此 $\ln(p_k/p_r)=z_k-z_r$。這表示**多項邏輯斯迴歸擬合的是各類相對於參考類別的對數機率比**。上例的超時分數比失敗高 1，所以超時的預測機率約是失敗的 $e^1\approx2.72$ 倍。

二元分類若輸出兩個 logits，正類機率等於對「正類分數減負類分數」做 Sigmoid。

**同一組 Softmax 機率，訓練和推論讀的位置不同。** 假設實際狀態是「超時」，訓練時就讀超時的機率約 0.245，交叉熵是 $-\ln(0.245)\approx1.41$。推論時依最高機率選類別，所以這筆會判成「正常」。Softmax 的三個輸出加總為 1，對應三個互斥的選項。

得到模型輸出後，仍要定義怎麼比較兩個模型。先用實數目標看為什麼正負殘差要轉成可加總的損失。


## 2. 真實答案、預測與誤差

| 記號 | 意思 |
|---|---|
| $x_i$ | 第 i 筆輸入特徵 |
| $y_i$ | 真實答案 |
| $\hat{y}_i$ | 預測值或預測類別，需交代定義 |
| $p_i$ | 第 i 筆屬於正類的預測機率 |
| $L$ | 損失，衡量預測和答案的不一致 |

若預測的是連續數值，可用平均平方誤差：

$$MSE=\frac1n\sum_i(y_i-\hat y_i)^2$$

**自編手算：**真實延遲 [10,20]，預測 [12,16]，誤差 [−2,4]，平方 [4,16]，平均為 10 ms²。損失是 10，不是「準確率 10%」。



## 3. 為什麼分類會用交叉熵？

### 先用通式回答「模型給答案多少機率」

對有 $K$ 個可能結果的問題，令 $q_k$ 為真實分布，$p_k$ 為模型分配給第 $k$ 個結果的機率。交叉熵（cross-entropy）的通式是

$H(q,p)=-\sum_{k=1}^K q_k\ln p_k.$

若一筆資料只有一個正確類別 $c$，則 $q_c=1$，其他項為 0，式子簡化為 $-\ln p_c$。例如服務的狀態只能在「正常、超時、失敗」中選一種，模型給這三類的機率為 $[0.1,0.8,0.1]$。實際狀態為超時時，損失為 $-\ln(0.8)\approx0.223$。其他類別仍會透過 Softmax 的分母影響分給超時的機率。

### BCE 如何由通式得到？Sigmoid 在哪一步？

BCE 是 binary cross-entropy（二元交叉熵）。把問題改成「是否超過 12 ms」後，令 $y=1$ 表示超時，$y=0$ 表示未超時。模型先算原始分數 $z$（logit），再用 Sigmoid 轉成超時機率：

$p=\sigma(z)=\frac{1}{1+e^{-z}}.$

以「超時、未超時」為排列順序，真實分布是 $(y,1-y)$，預測分布是 $(p,1-p)$。把這兩組值代入交叉熵通式，就得到

$L_{\mathrm{BCE}}=-[y\ln p+(1-y)\ln(1-p)].$

假設 $p=0.8$。若確實超時，BCE 取 $-\ln(0.8)\approx0.223$；若沒有超時，BCE 取 $-\ln(0.2)\approx1.609$。**Sigmoid 提供機率，BCE 根據真實答案評估機率。**

兩個類別互斥時，可以用一個 logit 搭配 Sigmoid＋BCE，也可以用兩個 logits 搭配 Softmax＋交叉熵。若 Softmax 的兩個分數分別為 $z_{\text{超時}}$ 和 $z_{\text{未超時}}$，超時機率等於 $\sigma(z_{\text{超時}}-z_{\text{未超時}})$。若同一筆請求**可以同時超時和發生辨識錯誤**，則每個標籤各有一個「是／否」分布，各算一次 BCE，再依設定加總或平均。

### 為什麼取對數？

BCE 和多類別交叉熵都取對數。本課使用自然對數，寫作 $\ln$；有些來源寫 $\log$，讀公式時先確認底數。

假設第一筆請求實際超時，第二筆實際未超時，而模型對兩筆都預測超時機率 0.8。模型分給兩個真實答案的機率分別是 0.8 和 0.2。若給定各筆輸入後，標籤在模型中可視為條件獨立，這批已觀測答案的概似就是 $0.8\times0.2=0.16$。取負對數後：

$-\ln(0.8\times0.2)=-\ln(0.8)-\ln(0.2)\approx1.833.$

**對數把相乘變成相加，負號把最大化概似變成最小化損失。** 固定同一批資料時，取平均也與取總和有相同的最小值位置。算對且把機率集中在真實答案上會降低損失。當真實答案只分到 1% 機率，單筆損失約為 4.605。若真實答案的機率趨近 0，負對數損失會持續增大。

**自我檢核。** 若模型給超時機率 0.8，但實際沒有超時，公式要用哪個機率？答案是 $1-0.8=0.2$，因為這筆實際觀測到的答案是「未超時」。

[來源：scikit-learn 的 log loss 說明](https://scikit-learn.org/stable/modules/model_evaluation.html#log-loss) · [PyTorch BCEWithLogitsLoss](https://docs.pytorch.org/docs/2.15/generated/torch.nn.BCEWithLogitsLoss.html) · [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/2.15/generated/torch.nn.CrossEntropyLoss.html)

### 完整手算｜用 10 筆資料估計超時機率

延續延遲服務。這裡用 10 筆自編紀錄，$x$ 是前方排隊數，$y=1$ 表示這筆實際用時超過 12 ms，$y=0$ 表示未超時。這次由原始紀錄算出係數，再代入一筆新請求。

#### 1. 算出輸入的平均與標準差

10 筆紀錄裡，5 筆的 $x=2$，另外 5 筆的 $x=6$。先用這批紀錄算平均數 $m$ 與除以筆數的標準差 $s_0$，沿用[第 04 課的標準化手算](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/04-scaling-normalization.md)。

$m=\frac{5(2)+5(6)}{10}=4,\qquad s_0=\sqrt{\frac{5(2-4)^2+5(6-4)^2}{10}}=2.$

所以 $x'=(x-4)/2$。排隊數 2 變成 $-1$，代表低於平均一個標準差。排隊數 6 變成 $1$，代表高於平均一個標準差。

#### 2. 從觀測結果算出模型係數

在 $x=2$ 的 5 筆請求中，只有 1 筆超時。若這一組的超時機率為 $p$，依前述的條件獨立假設，觀測到這 5 個標籤的概似是 $p(1-p)^4$，在 $p=1/5$ 時最大。同樣地，$x=6$ 的 5 筆裡有 4 筆超時，這組的概似在 $p=4/5$ 時最大。這兩個比例是**從實際標籤數出來的**。

| 排隊數 $x$ | 標準化後 $x'$ | 超時筆數／總筆數 | 擬合機率 $p$ | 由機率換成分數 $z=\ln[p/(1-p)]$ |
|---:|---:|---:|---:|---:|
| 2 | −1 | 1／5 | 20% | $\ln(0.2/0.8)=-\ln 4\approx-1.386$ |
| 6 | 1 | 4／5 | 80% | $\ln(0.8/0.2)=\ln 4\approx1.386$ |

模型要找一條 $z=b+wx'$ 的直線。$x'$ 從 $-1$ 到 $1$，增加了 2，分數從 $-\ln 4$ 到 $\ln 4$，增加了 $2\ln 4$，所以斜率是 $w=\ln 4$。直線在 $x'=0$ 時通過 $z=0$，所以截距 $b=0$。本例只有兩個輸入位置，直線的兩個係數剛好能讓兩組同時達到各自的概似最大值。輸入位置更多時，通常用數值最佳化一起估計係數。

代回 $x'=(x-4)/2$ 後，$z=(\ln 4)(x-4)/2=(x-4)\ln 2$，這就是第 1 節圖中的直線。兩種寫法分別用標準化後的尺度與原始尺度，對同一筆輸入會算出相同分數。

#### 3. 逐筆核對分數與機率

先用 $z=(\ln 4)x'$ 算分數，再用 $p=1/(1+e^{-z})$ 算超時機率。下表列出全部 10 筆。

| 筆次 | 排隊數 $x$ | 實際結果 $y$ | 標準化後 $x'$ | 分數 $z$ | 超時機率 $p$ |
|---:|---:|---:|---:|---:|---:|
| 1 | 2 | 0 | −1 | −1.386 | 20% |
| 2 | 2 | 0 | −1 | −1.386 | 20% |
| 3 | 2 | 0 | −1 | −1.386 | 20% |
| 4 | 2 | 0 | −1 | −1.386 | 20% |
| 5 | 2 | 1 | −1 | −1.386 | 20% |
| 6 | 6 | 0 | 1 | 1.386 | 80% |
| 7 | 6 | 1 | 1 | 1.386 | 80% |
| 8 | 6 | 1 | 1 | 1.386 | 80% |
| 9 | 6 | 1 | 1 | 1.386 | 80% |
| 10 | 6 | 1 | 1 | 1.386 | 80% |

第 5 筆的超時機率是 20%，實際仍然超時。第 6 筆的超時機率是 80%，實際卻未超時。當排隊數是模型唯一的輸入，同樣的排隊數就會得到同樣的預測機率。

這 10 筆中，有 8 筆的實際答案分到 0.8，另外 2 筆分到 0.2。把每筆的負對數相加再平均，得到本例的**擬合資料平均 BCE**：

$\frac{8[-\ln(0.8)]+2[-\ln(0.2)]}{10}\approx0.5004.$

#### 4. 算一筆新請求的超時機率

新請求的前方有 5 筆。沿用這批資料算出的 $m=4$、$s_0=2$ 和同一組模型係數：

$x'=\frac{5-4}{2}=0.5,\qquad z=(\ln 4)(0.5)=\ln 2,\qquad p=\frac{1}{1+e^{-\ln 2}}=\frac23\approx66.7\%.$

所以前方排 5 筆時，模型估計超時機率約為 **66.7%**。

### 補充｜先定義要預測的答案，再選輸出和損失

對同一套辨識服務，預測「延遲幾毫秒」與判斷「延遲是否超過 12 ms」是兩個不同的任務。前者的標籤是 5、7、9、14、19 ms，後者把同一批觀測改標成 0、0、0、1、1。**標籤的意義決定輸出的格式，損失則把任務在意的錯誤換成可最佳化的數值。**這個決定可以與隱藏層的啟動函數分開考慮。

| 任務與標籤 | 輸出如何解讀 | 常用損失與理由 |
|---|---|---|
| 預測實數延遲，例如 14 ms | 一個實數，迴歸常從恆等輸出開始，有硬性範圍時可另外設計 | MSE 加重大偏差，等變異數的獨立高斯誤差假設可導出最小平方。MAE 對偏差的懲罰是線性的，適合把條件中位數當目標。Huber 在小誤差下採平方，大誤差下近似線性，可依有意義的誤差尺度設定轉折點 |
| 是否超時，標籤為 0 或 1 | 一個 logit $z$，推論時以 sigmoid 解讀為正類分數 | BCE 對應二元結果的負對數概似。PyTorch 用 BCEWithLogitsLoss 直接接 logit |
| 一筆請求只屬於一種狀態 | $K$ 個 logits，softmax 後的機率總和為 1 | CrossEntropyLoss 直接接 logits。一般類別索引格式每筆只給一個整數類別 |
| 可能同時超時又發生辨識錯誤 | 每個標籤有一個 logit，各自轉成 sigmoid 分數 | BCEWithLogitsLoss 逐標籤計算。各分數不必加總為 1 |
| 給定前文，預測下一個 token | 每個位置輸出一組詞彙表 logits | token 級交叉熵，通常在需要監督的位置取平均 |

PyTorch 的 BCEWithLogitsLoss 已包含 sigmoid 的對應計算，CrossEntropyLoss 也直接處理 logits。訓練時把原始分數傳給這兩個 API，推論需要閱讀機率時才另外轉換。這是 API 的輸入契約。選擇警報閾值時，還要用驗證集檢查漏報與誤報的成本。若服務有少數真實的高延遲事件，應把那些事件的表現另外列出，避免單看一個平均損失。可接著閱讀 [機率與分類](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/02-probability-classification.md) 與 [分類模型評估](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/07-classification-evaluation.md)。

[來源：PyTorch BCEWithLogitsLoss](https://docs.pytorch.org/docs/2.15/generated/torch.nn.BCEWithLogitsLoss.html) · [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/2.15/generated/torch.nn.CrossEntropyLoss.html) · [Transformer 原論文 §3.4](https://arxiv.org/html/1706.03762v7#S3.SS4)

## 4. 相同 Accuracy，損失可能差很多

本教材程式的真實類別是 [0,1]；兩組正類機率分別為 [0.6,0.6] 與 [0.99,0.99]。以 0.5 為閾值，兩者都預測 [1,1]，所以 Accuracy 都是 50%。

第一組平均交叉熵約 0.714；第二組約 2.308。第二組對第一筆錯誤更自信，受到較大懲罰。請把手算結果對照 log_loss 輸出。



## 5. 模型怎麼學？怎麼連到 DLI？

模型學習是調整參數使訓練損失下降。梯度可先理解成「損失對某參數變動的局部敏感度」；以學習率控制每次調整幅度。下降的訓練損失不保證未見資料表現提升，仍要看驗證集。

下方用多項邏輯斯迴歸將每張 8×8 手寫數字影像分為 0 到 9 其中一類。程式使用 scikit-learn 的 digits 資料建立輕量基準，**不是 MNIST**。連到 DLI 時，請比較輸入形狀、輸出類別數與損失函數，避免把本例的分數當作 DLI 模型結果。

### 補充｜模型怎麼從損失知道每個權重該往哪裡改？

想像用 $\hat y=b+wx$ 預測延遲。若 $x=2$、$w=3$、$b=1$，模型預測 7 ms，實測 9 ms。殘差 $r=y-\hat y=2$ ms，單筆平方損失 $L=r^2=4$ ms²。

前向運算從 $x$ 算到 $\hat y$，再從 $\hat y$ 和答案算出 $L$。反向傳播（backpropagation）沿這條計算路徑反向套用鏈鎖律，算出損失對各參數的偏導數。此例中，$w$ 每變動一點，預測的變動率為 $x=2$。損失對預測的變動率為 $-2r=-4$。相乘得到：

$$\frac{\partial L}{\partial w}
=\frac{\partial L}{\partial \hat y}\frac{\partial\hat y}{\partial w}
=(-4)(2)=-8.$$

負梯度表示在目前這個點，小幅增加 $w$ 會使損失下降。若暫時固定 $b$，用梯度下降與學習率 0.1，權重更新為 $3-0.1(-8)=3.8$。新預測為 8.6 ms，單筆損失為約 0.16 ms²。**反向傳播負責算斜率，最佳化器根據斜率及其更新規則調整參數。** 在 PyTorch 中，常見的兩步是先呼叫 loss.backward()，再呼叫 optimizer.step()。

多層模型也遵循同一個原理。每向前回到一層，就再乘上該層的局部導數。下一課會說明多次相乘時，梯度為什麼可能變得很小。[第 06 課：Sigmoid 與梯度消失](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/06-overfitting-generalization.md) · [PyTorch 自動微分教學](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)

## 自我檢核

1. y=0，正類機率 p=0.9 時，應代入 −ln(0.9) 還是 −ln(0.1)？
2. Accuracy 不變時，交叉熵為何仍可能改善？
3. 訓練損失降低是否足以證明模型更適合部署？

### 參考解答

1. −ln(0.1)，因真實類別是負類。
2. 分類是否跨過閾值沒有改變，但給真實類別的機率可能更合適。
3. 否。還需驗證泛化、資料代表性及部署需求。

## 環境

若套件尚未安裝，先執行下一格。安裝需網路，之後的實驗不需下載資料。

In [ ]:
%pip -q install numpy matplotlib scikit-learn

### 迴歸手算驗證

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

queue = np.array([1., 2., 3.])
observed = np.array([2., 3., 7.])
design = np.column_stack([np.ones(queue.size), queue])
intercept, slope = np.linalg.lstsq(design, observed, rcond=None)[0]
prediction = intercept + slope * queue
residual = observed - prediction

print("Initial line MSE:", np.mean((observed - 2 * queue) ** 2))
print("OLS intercept and slope:", np.round([intercept, slope], 3))
print("Residuals and MSE:", np.round(residual, 3), np.mean(residual**2))
np.testing.assert_allclose(residual, [0.5, -1, 0.5])

fig, ax = plt.subplots()
ax.scatter(queue, observed, label="Observed")
ax.plot(queue, prediction, label="Fitted line")
ax.vlines(queue, np.minimum(observed, prediction),
          np.maximum(observed, prediction), linestyles="dashed", label="Residuals")
ax.set(xlabel="Requests ahead", ylabel="Latency (ms)")
ax.legend()
plt.show()

### 用 logits 驗證 BCE、兩類交叉熵與概似

兩筆請求的超時機率都為 0.8，第一筆未超時，第二筆超時。依序比較每筆的負對數損失、把同一分數改成兩個 logits 後的交叉熵，以及整批概似的負對數。預期三種算法的結果相符。

In [ ]:
import numpy as np

labels = np.array([0., 1.])
logits = np.log(np.array([4., 4.]))  # sigmoid(log(4)) = 0.8
probabilities = 1 / (1 + np.exp(-logits))
# Stable binary cross-entropy from logits: softplus(z) - y*z.
loss_per_example = np.logaddexp(0, logits) - labels * logits
print("Positive-class probabilities:", probabilities)
print("Binary targets and losses:", list(zip(labels.tolist(), loss_per_example.round(3).tolist())))
np.testing.assert_allclose(loss_per_example, -np.log([0.2, 0.8]))

# Two mutually exclusive classes, ordered as [not timed out, timed out].
two_class_logits = np.column_stack([np.zeros_like(logits), logits])
log_normalizer = np.logaddexp(two_class_logits[:, 0], two_class_logits[:, 1])
true_class_scores = two_class_logits[np.arange(labels.size), labels.astype(int)]
loss_from_two_class_ce = log_normalizer - true_class_scores

true_answer_prob = np.where(labels == 1, probabilities, 1 - probabilities)
joint_nll = -np.log(true_answer_prob.prod())
print("Two-class cross-entropy:", loss_from_two_class_ce.round(3))
print("Sum of BCE and joint negative log-likelihood:", loss_per_example.sum(), joint_nll)
np.testing.assert_allclose(loss_from_two_class_ce, loss_per_example)
np.testing.assert_allclose(joint_nll, loss_per_example.sum())

### 核對 10 筆資料的二元邏輯斯迴歸

用第 3 節的同一批紀錄算出尺度與係數，逐筆印出原始值、標籤、標準化值、分數與機率。最後核對平均 BCE 與新請求的超時機率。

In [ ]:
import numpy as np

queue_sizes = np.array([2, 2, 2, 2, 2, 6, 6, 6, 6, 6])
observed_timeout = np.array([0, 0, 0, 0, 1, 0, 1, 1, 1, 1])

queue_mean = queue_sizes.mean()
queue_scale = queue_sizes.std()
standardized_queue = (queue_sizes - queue_mean) / queue_scale

# 兩個排隊數的觀測超時比例，轉成直線上的兩個點。
levels = np.unique(standardized_queue)
group_rates = np.array([
    observed_timeout[standardized_queue == level].mean() for level in levels
])
group_scores = np.log(group_rates / (1 - group_rates))
slope = (group_scores[1] - group_scores[0]) / (levels[1] - levels[0])
intercept = group_scores[0] - slope * levels[0]

scores = intercept + slope * standardized_queue
probabilities = 1 / (1 + np.exp(-scores))
true_label_probabilities = np.where(observed_timeout == 1, probabilities, 1 - probabilities)
mean_bce = -np.log(true_label_probabilities).mean()

new_queue = 5
new_standardized_queue = (new_queue - queue_mean) / queue_scale
new_score = intercept + slope * new_standardized_queue
new_probability = 1 / (1 + np.exp(-new_score))

print(f"平均數 = {queue_mean:.0f}，標準差 = {queue_scale:.0f}")
print(f"截距 = {intercept:.3f}，斜率 = {slope:.3f}")
print("筆次 | 排隊數 | 實際結果 | 標準化後 | 分數 | 超時機率")
for i, (queue, actual, scaled, z, p) in enumerate(
    zip(queue_sizes, observed_timeout, standardized_queue, scores, probabilities), start=1
):
    print(f"{i:>2} | {queue} | {actual} | {scaled:>2.0f} | {z:>6.3f} | {p:.1%}")

print(f"平均 BCE = {mean_bce:.4f}")
print(f"新請求 x = {new_queue}，超時機率 = {new_probability:.1%}")
np.testing.assert_allclose(probabilities, np.repeat([0.2, 0.8], 5))
np.testing.assert_allclose(new_probability, 2 / 3)

In [ ]:
import numpy as np
from sklearn.metrics import log_loss, accuracy_score
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
truth = np.array([0, 1])
for name, prob in [('moderate', [0.6, 0.6]), ('high confidence', [0.99, 0.99])]:
    prob = np.array(prob)
    print(name, 'accuracy=', accuracy_score(truth, prob >= .5),
          'cross entropy=', log_loss(truth, np.column_stack([1-prob, prob])))
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
model.fit(X_train, y_train)
print('Digits baseline accuracy:', model.score(X_test, y_test))

## 如何解讀

scikit-learn digits 是 8×8 的資料，並非 DLI 使用的 MNIST；此處用它避免下載與 GPU 需求。這是基準示範，測試集不能反覆用於挑設定。

## 練習

把一個預測機率從 0.6 改成 0.99，觀察正確與錯誤時交叉熵的不同。

提交修改後的程式、結果圖表及 200 字解讀，列出至少一項限制。若用 AI 協助，說明你如何驗證結果。

## 連結原課程

回到 [DLI Notebook](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/tree/main/course_content/tutorials)，找出本概念對應的資料處理、模型訓練或評估步驟。

## 出處與閱讀建議

- **S5**：[scikit-learn，Logistic regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)。
- **S6**：[scikit-learn，Metrics and scoring（Log loss、Precision、Recall）](https://scikit-learn.org/stable/modules/model_evaluation.html)。

本課以所列來源的概念作繁體中文摘要，並加入自行編寫的情境、手算、程式及練習；不是原網站的逐段翻譯。標為「來源摘要」者為改寫，標為「自編」者為本教材教學設計；短引文另以引號及「中譯」標明。來源連結可供對照原文，查閱日期：2026-11-17。
